# E1 — Warm-start RL: does RL help *after* CE?

Cold-start RL fails at 0.5B (paper). Here we start RL from the CE-trained
policy (~0.98) and ask whether RL refinement adds points, holds, or
degrades it. Three arms from the released CE checkpoint: W1 warm+REINFORCE,
W2 warm+KL-REINFORCE (coef swept on seed 7), W3 warm+exact-PG.

**All shell calls use `os.system` with Python f-strings** so variables
interpolate reliably (a bash `!for` loop does NOT see Python vars).

Runtime -> T4/L4 GPU -> Run all. Download `warmstart_results.zip` at the end.

In [ ]:
import os, subprocess, json, torch
def run(cmd):
    rc = subprocess.call(cmd, shell=True)   # stream output live
    if rc != 0:
        raise RuntimeError(f'command failed (rc={rc}); see output above: {cmd[:80]}...')
    return rc
os.chdir('/content')
if not os.path.isdir('verifier-as-reward'):
    run('git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git')
os.chdir('/content/verifier-as-reward')
run('git pull --ff-only || true')
print('cwd:', os.getcwd())
assert torch.cuda.is_available(), 'enable a GPU runtime'
WARM = 'esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b'
run('PYTHONPATH=. python test_authority_verifier.py')
run('PYTHONPATH=. python make_expanded_train.py --train-seed 101 '
    '--train-traces-per-class 150 --val-seed 202 --val-traces-per-class 25')


## Step 1 — KL-coefficient sweep on seed 7 (pick the best on VAL)

In [ ]:
import os, json
WARM = 'esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b'
for kl in ['0.02', '0.1', '0.5']:
    print(f'=== KL={kl} ===')
    run(
        'PYTHONPATH=. python train_verifier_reward.py '
        '--model Qwen/Qwen2.5-0.5B '
        f'--warm-start-from {WARM} --kl-coef {kl} '
        '--balance-reward --prompt-style nl '
        '--steps 500 --batch-size 16 --lr 2e-5 '
        '--train-file expanded_train.jsonl --test-file expanded_val.jsonl '
        '--eval-every 25 --eval-max-actions 120 --seed 7 '
        f'--log-file training_log_warmstart_klsweep_{kl}.jsonl')
for kl in ['0.02', '0.1', '0.5']:
    pts = [json.loads(l) for l in open(f'training_log_warmstart_klsweep_{kl}.jsonl')
           if '\"step\"' in l]
    e = pts[-1]
    print(f"KL={kl}: final val acc {e['accuracy']:.3f} "
          f"viol {e['heldout_violation_rate']} fref {e['heldout_false_refuse_rate']}")

In [ ]:
BEST_KL = '0.1'   # <-- set from the sweep above (highest val acc, both errors low)
print('using KL =', BEST_KL)

## Step 2 — three arms x three seeds (train)

In [ ]:
import os
WARM = 'esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b'
arms = [('w1_reinforce', '--balance-reward'),
        ('w2_kl', f'--balance-reward --kl-coef {BEST_KL}'),
        ('w3_exactpg', '--exact-pg --balance-reward')]
for arm, extra in arms:
    for s in [7, 8, 9]:
        print(f'=== {arm} seed {s} ===')
        run(
            'PYTHONPATH=. python train_verifier_reward.py '
            '--model Qwen/Qwen2.5-0.5B '
            f'--warm-start-from {WARM} --prompt-style nl '
            '--steps 500 --batch-size 16 --lr 2e-5 '
            '--train-file expanded_train.jsonl --test-file expanded_val.jsonl '
            f'--eval-every 25 --eval-max-actions 120 --seed {s} {extra} '
            f'--log-file training_log_warmstart_{arm}_seed{s}.jsonl '
            f'--save-dir ckpt_warmstart_{arm}_seed{s}')

## Step 3 — evaluate every arm on committed test + fresh set (vs the CE start)

In [ ]:
# Fresh in-distribution set (new seed, deduped vs train) for a tighter CI.
from trace_benchmark import generate_corpus, write_jsonl, load_traces
from make_expanded_train import corpus_canonicals, drop_overlapping
test_canon = corpus_canonicals(load_traces('benchmark_test.jsonl'))
a, b = generate_corpus(101, 150)
train_canon = corpus_canonicals(drop_overlapping(a + b, test_canon, 'train'))
c, d = generate_corpus(770, 60)
fresh = drop_overlapping(c + d, train_canon | test_canon, 'fresh')
write_jsonl(fresh, 'warmstart_fresh.jsonl')
print('fresh actions:', sum(len(t['actions']) for t in fresh))

In [ ]:
import os, json
WARM = 'esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b'
json.dump({'backends': {}}, open('results_warmstart.json', 'w'))
def score(ckpt, tf):
    run('PYTHONPATH=. python train_verifier_reward.py '
                   f'--eval-checkpoint {ckpt} --test-file {tf} '
                   '--merge-results results_warmstart.json')
for tf in ['benchmark_test.jsonl', 'warmstart_fresh.jsonl']:
    score(WARM, tf)   # baseline row: the CE start itself
for arm in ['w1_reinforce', 'w2_kl', 'w3_exactpg']:
    for s in [7, 8, 9]:
        ck = f'ckpt_warmstart_{arm}_seed{s}'
        if os.path.isdir(ck):
            for tf in ['benchmark_test.jsonl', 'warmstart_fresh.jsonl']:
                score(ck, tf)
print('scored:', list(json.load(open('results_warmstart.json'))['backends']))

In [ ]:
import os
run('zip -j warmstart_results.zip training_log_warmstart_*.jsonl results_warmstart.json')
from google.colab import files
files.download('warmstart_results.zip')
